In [1]:
import sys
import numpy as np
import pybullet as p
import time
import logging
from typing import List, Tuple, Optional, Sequence, Collection, Dict, Any, cast
import random
import json

from predicators.structs import Action, Array, GroundAtom, Object, State, Type, ParameterizedOption
from predicators import utils
from predicators.settings import CFG
from gym.spaces import Box

#Import core environment methods, robot function etc.

from predicators.envs.pybullet_blocks import PyBulletBlocksEnv
from predicators.envs.pybullet_env import PyBulletEnv, create_pybullet_block
from predicators.envs.pybullet_multitable_blocks import PyBulletMultiTableBlocksEnv
from predicators.pybullet_helpers.robots import SingleArmPyBulletRobot
from predicators.pybullet_helpers.robots.mobile_single_arm import MobileSingleArmPyBulletRobot
from predicators.pybullet_helpers.geometry import Pose
from predicators.pybullet_helpers.joint import JointPositions, get_joint_infos, get_joint_positions
from predicators.pybullet_helpers.link import get_link_state, get_link_pose
from predicators.pybullet_helpers.controllers import create_base_reset_based_move_base_to_pick_option

pybullet build time: Jan 29 2025 23:16:28


In [2]:
from predicators.ground_truth_models.blocks.options import PyBulletMultiTableBlocksGroundTruthOptionFactory

In [3]:
PyBulletMultiTableBlocksGroundTruthOptionFactory._create_move_robot_base_to_pick_option

<bound method PyBulletMultiTableBlocksGroundTruthOptionFactory._create_move_robot_base_to_pick_option of <class 'predicators.ground_truth_models.blocks.options.PyBulletMultiTableBlocksGroundTruthOptionFactory'>>

In [3]:
logging.basicConfig(
    level=logging.WARNING,                    
    format="%(asctime)s %(name)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

#Defining test configuration, and overriding some default ones:
CFG.pybullet_robot = "fetch_mobile"
CFG.use_gui = True
CFG.env = "pybullet_multitable_blocks"
#Draws helpful debug lines in the workspace.
#NOT SURE WHETHER TO USE THIS. WILL DECIDE AFTER A COUPLE RUNS.
#CFG.pybullet_draw_debug = True
#Initializing with standard size of blocks.
CFG.blocks_block_size = 0.05
CFG.pybullet_birrt_num_iters = 50
CFG.pybullet_birrt_num_attempts = 10
CFG.pybullet_birrt_smooth_amt = 20
CFG.seed = random.randint(0,10000)
#CFG.seed = 12
#Num of PyBullet physics steps per high-level Action in visualize_action_sequence
CFG.pybullet_sim_steps_per_action = 200

In [4]:
multitable_env = PyBulletMultiTableBlocksEnv(use_gui=True)

In [5]:
multitable_env._table_ids

[2, 3, 4]

In [9]:
multitable_env._block_ids

[5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 29,
 30,
 31,
 32,
 33,
 34]

In [6]:
table_configs = {  
    0: {  # Table 0: Random piles  
        'exact_state': {},  # Not used for 'pile' mode  
        'setup': 'pile',  
        'params': [2, 3]  # 2 piles, 3 blocks per pile  
    },  
    1: {  # Table 1: Exact pile configuration  
        'exact_state': {  
            'pile1': ['red', 'blue', 'green'],  
            'pile2': ['yellow', 'purple'],  
            'pile3': ['orange']  
        },  
        'setup': 'exact_pile',  
        'params': None  
    },  
    2: {  # Table 2: Scattered blocks  
        'exact_state': ['cyan', 'magenta', 'lime', 'pink'],  
        'setup': 'exact_scattered',  
        'params': None  
    }  
}

In [7]:
multitable_env.set_state(table_configs)


Blocks with corresponding ids: {5: block0_0_0:block, 6: block0_0_1:block, 7: block0_0_2:block, 8: block0_1_0:block, 9: block0_1_1:block, 10: block0_1_2:block, 11: block1_1_0:block, 12: block1_1_1:block, 13: block1_1_2:block, 14: block1_2_3:block, 15: block1_2_4:block, 16: block1_3_5:block, 17: block2_1_0:block, 18: block2_2_1:block, 19: block2_3_2:block, 20: block2_4_3:block}.

Tables with corresponding ids: {2: table0:table, 3: table1:table, 4: table2:table}.


PyBulletState(data={table0:table: array([ 1. , -0.5,  0. ,  0. ], dtype=float32), table1:table: array([-1.7,  1.5,  0. ,  1. ], dtype=float32), table2:table: array([2.35, 2.  , 0.  , 2.  ], dtype=float32), robby:robot: array([1.15     , 1.       , 0.6999999, 1.       ], dtype=float32), block0_0_0:block: array([ 0.98777288, -0.59976027,  0.225     ,  0.        ,  0.15547769,
        0.18220576,  0.34647534]), block0_0_1:block: array([ 0.98777288, -0.59976027,  0.275     ,  0.        ,  0.57218808,
        0.54713927,  0.85117486]), block0_0_2:block: array([ 0.98777288, -0.59976027,  0.325     ,  0.        ,  0.5717438 ,
        0.64526642,  0.10054279]), block0_1_0:block: array([ 0.90165976, -0.49632033,  0.225     ,  0.        ,  0.64288129,
        0.34998227,  0.59912875]), block0_1_1:block: array([ 0.90165976, -0.49632033,  0.275     ,  0.        ,  0.63969684,
        0.43650598,  0.72195001]), block0_1_2:block: array([ 0.90165976, -0.49632033,  0.325     ,  0.        ,  0.43180432

In [26]:
multitable_env._table_workspaces

[{'x_lb': 0.875, 'x_ub': 1.125, 'y_lb': -0.7, 'y_ub': -0.3},
 {'x_lb': -1.825, 'x_ub': -1.575, 'y_lb': 1.3, 'y_ub': 1.7},
 {'x_lb': 2.225, 'x_ub': 2.475, 'y_lb': 1.8, 'y_ub': 2.2}]

In [8]:
multitable_env._pybullet_robot

In [9]:
#Create a block type and table type
type_dict = {t.name: t for t in multitable_env.types}

In [10]:
type_dict['block']

Type(name='block')

In [11]:
block_to_pick_object = Object("block2_1_0", type_dict['block'])

In [12]:
block_to_pick_object.name

'block2_1_0'

In [13]:
table_to_pick_from_object = Object("table2", type_dict['table'])
robot_object = Object("robby", type_dict['robot'])

In [14]:
move_option = PyBulletMultiTableBlocksGroundTruthOptionFactory._create_move_robot_base_to_pick_option(name='MoveRobotToPick', 
                                                                robot=multitable_env._pybullet_robot, option_types=[type_dict['robot'],
                                                                type_dict['block']], params_space=Box(0, 1, (0, )), env=multitable_env, 
                                                                physics_client_id=multitable_env._physics_client_id)

In [15]:
grounded_move_option=move_option.ground([robot_object, block_to_pick_object], params=np.array([], dtype=np.float32))

In [16]:
state = multitable_env._current_observation

In [17]:
memory = {}
move_base_waypoints = []

In [18]:
assert grounded_move_option.initiable(state)

In [19]:
try:
    while not grounded_move_option.terminal(state):
        waypoint = grounded_move_option.policy(state)
        move_base_waypoints.append(waypoint)
except utils.OptionExecutionFailure as e:
    logging.error(f"\nExecuting move option for picking failed.")
        

Target EE position for block block2_1_0: Pose(position=(2.3544872554292073, 1.8270232703787177, 0.42500000000000004), orientation=(0.7071, 0.0, -0.7071, 0.0)).

IK Succeeded for Target EE position: Pose(position=(2.3544872554292073, 1.8270232703787177, 0.625), orientation=(0.7071, 0.0, -0.7071, 0.0)) at test pose: (2.877044759526159, 1.3389231738374552, 2.3902753358662387).


In [20]:
move_base_waypoints

[Action(_arr=array([ 0.55694601, -0.799964  ,  1.91205568, -1.38951883,  0.7206384 ,
        -1.47059617, -0.64174952,  0.04      ,  0.04      ]), extra_info={'base_motion': {'mode': 'smooth_position', 'params': (0.55, 1.0, 0.0)}}),
 Action(_arr=array([ 0.55694601, -0.799964  ,  1.91205568, -1.38951883,  0.7206384 ,
        -1.47059617, -0.64174952,  0.04      ,  0.04      ]), extra_info={'base_motion': {'mode': 'smooth_position', 'params': (0.5569963781679459, 1.0071985631805318, -0.006477018534734036)}}),
 Action(_arr=array([ 0.55694601, -0.799964  ,  1.91205568, -1.38951883,  0.7206384 ,
        -1.47059617, -0.64174952,  0.04      ,  0.04      ]), extra_info={'base_motion': {'mode': 'smooth_position', 'params': (0.5639927563358917, 1.0143971263610636, -0.012954037069468516)}}),
 Action(_arr=array([ 0.55694601, -0.799964  ,  1.91205568, -1.38951883,  0.7206384 ,
        -1.47059617, -0.64174952,  0.04      ,  0.04      ]), extra_info={'base_motion': {'mode': 'smooth_position', 'para